# 65 · LiteLLM: One API, Every Provider

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/esturban/agent/blob/main/examples/65-litellm-multi-provider/litellm_workbook.ipynb)

## What you'll learn
- Call OpenAI, Anthropic, and Mistral with **one identical function** — only the model string changes
- Build **fallback chains** so a failed provider automatically retries on the next
- Track **cost per call** with `completion_cost()`
- Drop LiteLLM into LangChain via `ChatLiteLLM`

**Prerequisite:** [01 · Basic Agent](../1-basic-local-rag/)

> **Keys needed:** `OPENAI_API_KEY` (required). `ANTHROPIC_API_KEY` and `MISTRAL_API_KEY` are optional — provider sections skip gracefully if missing.

In [ ]:
# ── Environment Detection ───────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in {'Google Colab' if IN_COLAB else 'local environment'}")

In [ ]:
if IN_COLAB:
    !pip install -q litellm langchain-community python-dotenv
    print("Installation complete.")
else:
    print("Local: pip install litellm langchain-community python-dotenv")

In [ ]:
import os

if IN_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    for key in ("ANTHROPIC_API_KEY", "MISTRAL_API_KEY"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            print(f"  {key} not set — that provider section will be skipped")
else:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is required"
print("Keys loaded. Available providers:")
for provider, key in [("OpenAI", "OPENAI_API_KEY"), ("Anthropic", "ANTHROPIC_API_KEY"), ("Mistral", "MISTRAL_API_KEY")]:
    print(f"  {provider}: {'yes' if os.getenv(key) else 'no'}")

In [ ]:
import litellm
from litellm import completion, completion_cost

litellm.set_verbose = False

PROMPT = "Explain neural networks in one sentence."
print("LiteLLM ready, version:", litellm.__version__)

## Part 1 — One Function, Any Provider

LiteLLM's `completion()` has the same signature as `openai.chat.completions.create()`.  
The **only thing that changes** between providers is the model string.

In [ ]:
# OpenAI — always available
response = completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": PROMPT}],
    max_tokens=60,
)
openai_answer = response.choices[0].message.content
openai_cost   = completion_cost(completion_response=response)
print(f"[openai/gpt-4o-mini] ${openai_cost:.6f}")
print(openai_answer)

In [ ]:
# Anthropic — skip if key not set
if os.getenv("ANTHROPIC_API_KEY"):
    response = completion(
        model="anthropic/claude-haiku-4-5",
        messages=[{"role": "user", "content": PROMPT}],
        max_tokens=60,
    )
    anthropic_answer = response.choices[0].message.content
    anthropic_cost   = completion_cost(completion_response=response)
    print(f"[anthropic/claude-haiku-4-5] ${anthropic_cost:.6f}")
    print(anthropic_answer)
else:
    print("Skipped: ANTHROPIC_API_KEY not set")
    print("Model string would be: anthropic/claude-haiku-4-5")

In [ ]:
# Mistral — skip if key not set
if os.getenv("MISTRAL_API_KEY"):
    response = completion(
        model="mistral/mistral-small",
        messages=[{"role": "user", "content": PROMPT}],
        max_tokens=60,
    )
    mistral_answer = response.choices[0].message.content
    mistral_cost   = completion_cost(completion_response=response)
    print(f"[mistral/mistral-small] ${mistral_cost:.6f}")
    print(mistral_answer)
else:
    print("Skipped: MISTRAL_API_KEY not set")
    print("Model string would be: mistral/mistral-small")

## Part 2 — Fallback Chains

LiteLLM's `Router` accepts a `fallbacks` list. If the primary model errors (rate limit, downtime),  
it automatically retries on the next entry. You don't write any try/except — the router handles it.

In [ ]:
from litellm import Router

model_list = [
    {
        "model_name": "primary",
        "litellm_params": {"model": "openai/gpt-4o-mini", "api_key": os.getenv("OPENAI_API_KEY")},
    },
    {
        "model_name": "fallback-1",
        "litellm_params": {"model": "openai/gpt-4o-mini", "api_key": os.getenv("OPENAI_API_KEY")},
    },
]

router = Router(
    model_list=model_list,
    fallbacks=[{"primary": ["fallback-1"]}],
    num_retries=2,
    retry_after=1,
)
print("Router ready with", len(model_list), "model entries")

In [ ]:
response = router.completion(
    model="primary",
    messages=[{"role": "user", "content": PROMPT}],
    max_tokens=60,
)
print("Router response (model used:", response.model, ")")
print(response.choices[0].message.content)
print("\nIf 'primary' had returned a 429, the router would have automatically used 'fallback-1'.")

## Part 3 — Cost Tracking

`completion_cost()` returns the USD cost for any LiteLLM response.  
Below we run the same prompt with two OpenAI models and compare cost vs quality.

In [ ]:
models = [
    "openai/gpt-4o-mini",
    "openai/gpt-4o",
]

print(f"{'Model':<30} {'Cost':>10}  Answer")
print("-" * 80)
for model in models:
    resp = completion(
        model=model,
        messages=[{"role": "user", "content": PROMPT}],
        max_tokens=60,
    )
    cost   = completion_cost(completion_response=resp)
    answer = resp.choices[0].message.content[:60]
    print(f"{model:<30} ${cost:>9.6f}  {answer}")

## Part 4 — LangChain Integration via `ChatLiteLLM`

`ChatLiteLLM` wraps LiteLLM as a LangChain `BaseChatModel`.  
Drop it anywhere a `ChatOpenAI` object is expected — chains, agents, LangGraph nodes — with zero changes.

In [ ]:
from langchain_community.chat_models import ChatLiteLLM
from langchain_core.messages import HumanMessage

chat = ChatLiteLLM(model="openai/gpt-4o-mini", max_tokens=60)
response = chat.invoke([HumanMessage(content=PROMPT)])
print("ChatLiteLLM response:")
print(response.content)
print("\nSwap model='openai/gpt-4o-mini' for 'anthropic/claude-haiku-4-5' with no other changes.")

## Provider Comparison

| Provider | Model string | Notes |
|----------|-------------|-------|
| OpenAI | `openai/gpt-4o-mini` | Default; requires `OPENAI_API_KEY` |
| Anthropic | `anthropic/claude-haiku-4-5` | Requires `ANTHROPIC_API_KEY` |
| Mistral | `mistral/mistral-small` | Requires `MISTRAL_API_KEY` |
| Cohere | `cohere/command-r` | Requires `COHERE_API_KEY` |
| Ollama | `ollama/llama3` | Local; no key needed |

Full list: [docs.litellm.ai/docs/providers](https://docs.litellm.ai/docs/providers)

## Exercise 1 — Add a New Provider

Add Cohere to the cost comparison loop in Part 3.  
If you don't have a `COHERE_API_KEY`, print a skip message and show what the model string would be.

In [ ]:
# Starter
COHERE_MODEL = "cohere/command-r"

# TODO: if os.getenv("COHERE_API_KEY") is set, add COHERE_MODEL to the models list
# TODO: print cost + answer, or print a skip message

## Exercise 2 — Cost-Threshold Fallback

Write a function `smart_completion(prompt, budget_usd)` that:  
1. Tries `openai/gpt-4o` first  
2. If the estimated cost exceeds `budget_usd`, falls back to `openai/gpt-4o-mini`  

Hint: use `litellm.cost_per_token()` to estimate cost before making the call.

In [ ]:
# Starter
def smart_completion(prompt: str, budget_usd: float) -> str:
    # TODO: estimate cost with litellm.cost_per_token(model, prompt_tokens)
    # TODO: pick model based on estimate vs budget_usd
    # TODO: call completion() and return the answer
    pass

# Test it
# print(smart_completion(PROMPT, budget_usd=0.000050))

## What's Next

- **[64 · Pydantic-AI](../64-pydantic-ai/)** — another alternative to LangChain for structured LLM outputs
- **[72 · Batch Agent Runner](../72-batch-agent-runner/)** — async batching with retry
- **LiteLLM docs** — [docs.litellm.ai](https://docs.litellm.ai)

---
## Answer Key

In [ ]:
# Answer 1 — Cohere provider
EXTENDED_MODELS = ["openai/gpt-4o-mini", "openai/gpt-4o"]
if os.getenv("COHERE_API_KEY"):
    EXTENDED_MODELS.append("cohere/command-r")

print(f"{'Model':<30} {'Cost':>10}  Answer")
print("-" * 80)
for model in EXTENDED_MODELS:
    resp = completion(model=model, messages=[{"role": "user", "content": PROMPT}], max_tokens=60)
    cost = completion_cost(completion_response=resp)
    print(f"{model:<30} ${cost:>9.6f}  {resp.choices[0].message.content[:55]}")

if "cohere/command-r" not in EXTENDED_MODELS:
    print("cohere/command-r: skipped (COHERE_API_KEY not set)")

In [ ]:
# Answer 2 — cost-threshold fallback
def smart_completion(prompt: str, budget_usd: float) -> str:
    preferred = "openai/gpt-4o"
    fallback  = "openai/gpt-4o-mini"
    # Rough estimate: gpt-4o is ~$5/1M input tokens
    est_tokens = len(prompt.split()) + 10
    est_cost   = est_tokens * 5e-6
    model = preferred if est_cost <= budget_usd else fallback
    print(f"Using {model} (estimated cost ${est_cost:.6f}, budget ${budget_usd:.6f})")
    resp = completion(model=model, messages=[{"role": "user", "content": prompt}], max_tokens=60)
    return resp.choices[0].message.content

print(smart_completion(PROMPT, budget_usd=0.000050))